# AdaBoost From Scratch

This notebook is a pedagogical Python reimplementation of `matlab/ada-boost/AdaBoost.m`.

We keep the spirit of the original demo (2D synthetic data + weak linear separators), while making the derivation and visuals easier to follow.

## Mathematical Setup

Given training samples $(x_i, y_i)$ with $y_i\in\{-1,+1\}$ and weak learners $h_j(x)\in\{-1,+1\}$, AdaBoost builds a score

$$
F_T(x) = \sum_{t=1}^T \alpha_t h_{j_t}(x),
$$

and predicts $\mathrm{sign}(F_T(x))$.

At each round $t$, we choose the weak learner with smallest weighted error

$$
\varepsilon_t = \sum_i D_t(i)\,\mathbf{1}[h_{j_t}(x_i)\neq y_i],
$$

then use

$$
\alpha_t = \frac12\log\frac{1-\varepsilon_t}{\varepsilon_t},
\qquad
D_{t+1}(i) \propto D_t(i)\exp\big(-\alpha_t y_i h_{j_t}(x_i)\big).
$$

The algorithm greedily decreases an exponential upper bound of the empirical risk.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from ipywidgets import interact, IntSlider

plt.rcParams['figure.dpi'] = 120
rng = np.random.default_rng(2)

## 1) Build the Toy Dataset (ring vs. blob)

In [ ]:
n = 1000
n_half = n // 2

# Ring class
theta = 2 * np.pi * rng.random(n_half)
R = 0.95
r = R * (1 + 0.08 * rng.standard_normal(n_half))
x_ring = np.c_[np.cos(theta) * r, np.sin(theta) * r]

# Gaussian blob class
x_blob = 0.25 * rng.standard_normal((n_half, 2))

X = np.vstack([x_blob, x_ring])
y = np.r_[np.ones(n_half), -np.ones(n_half)]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[y > 0, 0], X[y > 0, 1], s=10, c='crimson', label='+1 class')
ax.scatter(X[y < 0, 0], X[y < 0, 1], s=10, c='royalblue', label='-1 class')
ax.set_title('Synthetic dataset')
ax.set_aspect('equal')
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.legend(loc='upper right')
plt.show()

## 2) Create Weak Learners (random oriented linear separators)

In [ ]:
r = 120  # number of weak learners
angles = 2 * np.pi * rng.random(r)
u = np.vstack([np.cos(angles), np.sin(angles)])  # normal vectors
t = 1.2 * (2 * rng.random(r) - 1)  # offsets

# Raw predictions of all weak learners on all points: shape (n, r)
raw = X @ u - t

# Flip sign if needed so each learner is (weakly) correlated with labels
acc = (np.sign(raw) == y[:, None]).mean(axis=0)
sigma = np.sign(acc - 0.5 + 1e-10)
H = np.sign(raw * sigma[None, :])

# Avoid exact zeros from sign(0)
H[H == 0] = 1
E = (H != y[:, None]).astype(float)
print('Weak learner weighted-error range (uniform weights):', E.mean(axis=0).min(), E.mean(axis=0).max())

## 3) Visualize a few weak decision boundaries

In [ ]:
s = np.linspace(-1.2, 1.2, 300)
U, V = np.meshgrid(s, s)
Z = np.c_[U.ravel(), V.ravel()]

fig, axes = plt.subplots(2, 3, figsize=(10, 6), constrained_layout=True)
for k, ax in enumerate(axes.flat):
    if k >= 6:
        break
    pred = np.sign((Z @ u[:, k] - t[k]) * sigma[k]).reshape(U.shape)
    pred[pred == 0] = 1
    ax.contourf(U, V, pred, levels=[-1, 0, 1], cmap='Greys', alpha=0.35)
    ax.scatter(X[y > 0, 0], X[y > 0, 1], s=6, c='crimson')
    ax.scatter(X[y < 0, 0], X[y < 0, 1], s=6, c='royalblue')
    ax.set_aspect('equal')
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_title(f'Weak learner #{k+1}')
plt.show()

## 4) AdaBoost Loop

In [ ]:
def run_adaboost(H, E, y, n_iter=400):
    n, r = H.shape
    w = np.zeros(r)
    D = np.ones(n) / n
    exp_losses = []
    learner_ids = []
    alphas = []
    D_hist = [D.copy()]

    for _ in range(n_iter):
        F = H @ w
        exp_losses.append(np.mean(np.exp(-y * F)))

        weighted_errors = (D[:, None] * E).sum(axis=0)
        j = np.argmin(weighted_errors)
        eps = np.clip(weighted_errors[j], 1e-8, 1 - 1e-8)
        a = 0.5 * np.log((1 - eps) / eps)

        w[j] += a
        D *= np.exp(-a * y * H[:, j])
        D /= D.sum()

        learner_ids.append(int(j))
        alphas.append(float(a))
        D_hist.append(D.copy())

    return {
        'w': w,
        'exp_losses': np.array(exp_losses),
        'learner_ids': np.array(learner_ids),
        'alphas': np.array(alphas),
        'D_hist': D_hist,
    }

boost = run_adaboost(H, E, y, n_iter=500)
print('Training rounds:', len(boost['exp_losses']))

## 5) Loss decay and weighted sample focus

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(boost['exp_losses'], lw=2)
ax.set_yscale('log')
ax.set_xlabel('Iteration')
ax.set_ylabel('Exponential loss (log scale)')
ax.set_title('AdaBoost optimization progress')
ax.grid(alpha=0.25)
plt.show()

D0 = boost['D_hist'][0]
Dend = boost['D_hist'][-1]
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, Dcur, title in zip(axes, [D0, Dend], ['Initial sample weights', 'Final sample weights']):
    sizes = 20 + 400 * (Dcur / Dcur.max())
    ax.scatter(X[y > 0, 0], X[y > 0, 1], s=sizes[y > 0], c='crimson', alpha=0.7)
    ax.scatter(X[y < 0, 0], X[y < 0, 1], s=sizes[y < 0], c='royalblue', alpha=0.7)
    ax.set_aspect('equal')
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_title(title)
plt.show()

## 6) Interactive decision map over iterations

In [ ]:
grid_scores = []
W = np.zeros_like(boost['w'])
for j, a in zip(boost['learner_ids'], boost['alphas']):
    W[j] += a
    g = np.sign((Z @ u - t) * sigma).astype(float)
    g[g == 0] = 1
    grid_scores.append((g @ W).reshape(U.shape))

def show_round(it=1):
    score = grid_scores[it - 1]
    D = boost['D_hist'][it]

    fig, ax = plt.subplots(figsize=(6, 6))
    norm = TwoSlopeNorm(vmin=score.min(), vcenter=0.0, vmax=score.max())
    im = ax.contourf(U, V, score, levels=25, cmap='RdBu_r', norm=norm, alpha=0.9)
    ax.contour(U, V, score, levels=[0], colors='k', linewidths=2)
    sizes = 12 + 120 * (D / D.max())
    ax.scatter(X[y > 0, 0], X[y > 0, 1], s=sizes[y > 0], c='white', edgecolor='crimson', linewidth=1)
    ax.scatter(X[y < 0, 0], X[y < 0, 1], s=sizes[y < 0], c='white', edgecolor='royalblue', linewidth=1)
    ax.set_aspect('equal')
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_title(f'AdaBoost decision score at iteration {it}')
    fig.colorbar(im, ax=ax, shrink=0.85)
    plt.show()

interact(show_round, it=IntSlider(min=1, max=len(grid_scores), step=1, value=50));

## 7) Takeaways

- Even very simple weak learners (random oriented lines) can build nonlinear boundaries.
- Sample weights naturally concentrate on hard examples near the class overlap.
- The exponential loss decays quickly, matching the classic AdaBoost theory.